# Ribosome Network — Chaperone (validator) on Kaggle GPU

Runs the **validator side** offline: refold oracle benchmarking, **GPU exact
k-mer Jaccard duplicate detection** (θ_dup = 0.85), and a full adversarial
epoch with the production `Chaperone` pipeline — including the known attack
outcomes (Sybil clones zeroed, leakers rejected).

**Kaggle setup**: GPU **T4 x2** or **RTX Pro 6000** · Internet **ON** ·
add the `ribosome-network` dataset (or set `REPO_URL`).


In [ ]:
# --- 0. Environment probe -------------------------------------------------
# Verify the accelerator before anything else. Expected on Kaggle:
#   GPU T4 x2      -> 2 devices, 16 GiB each  (kernels run one device each)
#   RTX Pro 6000   -> 1 device, ~96 GiB       (kernels share, larger batches)
import subprocess, sys, platform, json, time

print("python", sys.version.split()[0], "|", platform.platform())
try:
    nvidia = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, timeout=20).stdout.strip()
    print("nvidia-smi:", nvidia.replace("\n", " | ") or "(none)")
except FileNotFoundError:
    nvidia = ""
    print("nvidia-smi not found - switch the notebook Accelerator to GPU!")

import torch
n_dev = torch.cuda.device_count()
print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()} | devices: {n_dev}")

if torch.cuda.is_available():
    DEVICES = [f"cuda:{i}" for i in range(n_dev)]
    for i in range(n_dev):
        p = torch.cuda.get_device_properties(i)
        print(f"  cuda:{i} -> {p.name}, {p.total_memory/2**30:.1f} GiB")
else:
    DEVICES = ["cpu"]
    print("WARNING: no CUDA - kernels fall back to CPU (slow but correct)")
GPU_MEM_GIB = (torch.cuda.get_device_properties(0).total_memory / 2**30
               if torch.cuda.is_available() else 0.0)
IS_T4 = "T4" in (nvidia or "")
print("device plan:", DEVICES)


In [ ]:
# --- 1. Dependencies ------------------------------------------------------
# Mechanism core needs only numpy; ViennaRNA ships as a pip wheel (folds via
# bundled libRNA, no conda needed on Kaggle). bittensor is NOT needed here:
# these notebooks exercise the same mechanism code path offline that the
# live neurons run on testnet.
import subprocess, sys

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout

sh(f"{sys.executable} -m pip install -q numpy pytest viennarna")

try:
    import ViennaRNA
    print("ViennaRNA wheel OK - physics-grade 2D oracle available")
except ImportError:
    print("ViennaRNA unavailable - notebooks will use Nussinov oracle")

import numpy, pytest
print("numpy", numpy.__version__, "| pytest", pytest.__version__)


In [ ]:
# --- 2. Package bootstrap -------------------------------------------------
# Find the ribosome-network package. Two supported paths:
#   a) Kaggle Dataset: upload ribosome-network.zip (or the folder) as an
#      input ("Add Input" -> your dataset). This cell locates and extracts it.
#   b) git clone: set REPO_URL below to your GitHub repo and run once.
import glob, zipfile, shutil, sys, os
from pathlib import Path

REPO_URL = "https://github.com/RibosomeNetwork/ribosome-network"  # <- your fork
WORK = Path("/kaggle/working")
candidates = (glob.glob("/kaggle/input/**/*ribosome*", recursive=True)
              + glob.glob("/kaggle/input/*/*.zip"))
target = None
for c in candidates:
    if c.endswith(".zip") and "ribosome" in c.lower():
        target = c
        with zipfile.ZipFile(c) as z:
            z.extractall(WORK / "pkg")
        break
if target is None and candidates:
    target = candidates[0]  # a dataset directory

root = None
for base in ([WORK / "pkg"] + [Path(c) for c in candidates]):
    if base is None:
        continue
    for p in [base, *base.glob("**/ribosome")]:
        if p.name == "ribosome" and p.is_dir():
            root = p.parent
            break
    if root:
        break

if root is None:
    print("no Kaggle dataset found - cloning", REPO_URL)
    os.system(f"git clone -q {REPO_URL} {WORK/'ribosome-network'}")
    root = WORK / "ribosome-network"

sys.path.insert(0, str(root))
os.chdir(root)
print("package root:", root)

from ribosome import constants  # noqa: E402
print("mechanism constants: theta_dup=%.2f w_div=%.1f T_rot=%d B=%d K=%d"
      % (constants.THETA_DUP, constants.W_DIV, constants.T_ROT,
         constants.SCORE_REVEAL_DELAY_B, constants.K_CANDIDATES))


In [ ]:
# --- 3. Sanity: run the mechanism test-suite ------------------------------
# 92 tests, ~10 s CPU. If this is green, the Kaggle runtime executes the
# exact code path the testnet neurons use.
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "--no-header"],
                   capture_output=True, text=True, timeout=600)
print(r.stdout[-1200:])
assert " failed" not in r.stdout.splitlines()[-1], "test-suite failed!"


## Oracle batch benchmark

The chaperone refolds every revealed candidate. Throughput decides how many
miners one validator can handle within the 6-minute EVALUATE phase.


In [ ]:
# --- 4. Oracle throughput ----------------------------------------------------
import time, random, json
from ribosome.oracle import StubOracle
from ribosome.rna import random_sequence

rng = random.Random(2)
seqs = [random_sequence(rng, 110) for _ in range(200)]
res = {}
stub = StubOracle()
t0 = time.perf_counter(); [stub.fold(s) for s in seqs]
res["nussinov"] = len(seqs) / (time.perf_counter() - t0)
print(f"Nussinov  {res['nussinov']:9.1f} seq/s (memoized)")
try:
    from ribosome.oracle import PyViennaRNAOracle
    vr = PyViennaRNAOracle()
    t0 = time.perf_counter(); [vr.fold(s) for s in seqs]
    res["viennarna"] = len(seqs) / (time.perf_counter() - t0)
    print(f"ViennaRNA {res['viennarna']:9.1f} seq/s")
except ImportError:
    print("ViennaRNA unavailable")
worst = min(res.values())
print(f"-> EVALUATE budget 360s covers ~{worst*360:,.0f} refolds "
      f"(= {int(worst*360/4):,} miners at K=4)")
json.dump(res, open("oracle_rates.json", "w"), indent=2)


## GPU duplicate detection (exact k-mer Jaccard, θ_dup = 0.85)

Duplicate detection shingles each best candidate into 3-mers and compares
**sets** (Jaccard). With k = 3 there are only 4³ = 64 possible shingles, so
each sequence is a 64-dim boolean vector and the *full pairwise Jaccard
matrix* is two matmuls — exact, and trivially GPU-shaped. We scale to
thousands of miners to prove the θ_dup gate holds far beyond the 32-uid
testnet size, with mutated clones planted as ground truth.


In [ ]:
# --- 5. GPU k-mer Jaccard -----------------------------------------------------
import torch, random, time
import numpy as np

def shingle_matrix(seqs, k=3):
    """[N, 64] boolean k-mer-set matrix (exact for k=3)."""
    idx = {"".join(b): i for i, b in enumerate(
        __import__("itertools").product("AUGC", repeat=k))}
    m = np.zeros((len(seqs), 64), dtype=np.float32)
    for r, s in enumerate(seqs):
        s = s.upper().replace("T", "U")
        for i in range(len(s) - k + 1):
            m[r, idx[s[i:i+k]]] = 1.0
    return torch.from_numpy(m)

def gpu_jaccard(seqs, device=DEVICES[0], k=3):
    m = shingle_matrix(seqs, k).to(device)
    inter = m @ m.T
    norms = m.sum(1, keepdim=True)
    union = norms + norms.T - inter
    return (inter / union.clamp(min=1)).cpu()

# population with planted clones: copies of base miners at controlled
# mutation distances (1-3 point mutations of a 110nt sequence)
rng = random.Random(5)
base = [random_sequence(rng, 110) for _ in range(384)]
clones, mut_of = [], []
for muts in (1, 2, 3):
    for _ in range(96):
        s = list(base[rng.randrange(len(base))])
        for _ in range(muts):
            s[rng.randrange(len(s))] = rng.choice("AUGC")
        clones.append("".join(s))
        mut_of.append(muts)
pop = base + clones
clone_slice = slice(len(base), len(pop))

t0 = time.perf_counter(); J = gpu_jaccard(pop); gpu_dt = time.perf_counter() - t0
N = len(pop)
Jn = J.numpy()
dup_pairs = [(i, j) for i in range(N) for j in range(i+1, N) if Jn[i, j] >= 0.85]
caught = {i for pr in dup_pairs for i in pr} & set(range(len(base), N))
print(f"[{DEVICES[0]}] {N}x{N} Jaccard in {gpu_dt*1000:.0f} ms; "
      f"pairs >= theta_dup: {len(dup_pairs)}")
for muts in (1, 2, 3):
    idx = [len(base) + j for j, m in enumerate(mut_of) if m == muts]
    rec = sum(1 for i in idx if i in caught) / len(idx)
    print(f"clone recall @ {muts} mutation(s): {100*rec:5.1f}%")
assert sum(1 for i in caught if mut_of[i - len(base)] == 1) / 96 >= 0.95
assert sum(1 for i in caught if mut_of[i - len(base)] == 2) / 96 >= 0.85
print("gate holds: single/double mutants are caught at theta_dup = 0.85;")
print("triple mutants leak by design (that is what w_div + rotation are for)")

# CPU comparison
t0 = time.perf_counter(); _ = shingle_matrix(pop) @ shingle_matrix(pop).T
cpu_dt = time.perf_counter() - t0
print(f"CPU same op: {cpu_dt*1000:.0f} ms -> speedup x{cpu_dt/max(gpu_dt,1e-9):.1f}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.2), constrained_layout=True)
off = Jn[np.triu_indices(N, 1)]
ax.hist(off[off < 0.99], bins=60, color="#3a6ea5")
ax.axvline(0.85, color="crimson", ls="--", label=r"$\theta_{dup}$ = 0.85")
ax.set_xlabel("k-mer Jaccard"); ax.set_ylabel("pairs"); ax.legend()
ax.set_title("pairwise duplicate similarity (384 unique + 128 clones)")
plt.savefig("gpu_duplicates.png", dpi=140); plt.show()


## Full adversarial epoch (production Chaperone pipeline)

32-miner population with the same attack mix as the simulation evidence:
honest GA + stub miners, **Sybil clones** (replay another miner's reveal),
**lazy** (withhold reveals), **leakers** (commit to targets outside the pool).
Expected: clones 98–99% zeroed, leakers 100% rejected, honest miners earn
proportional weights.


In [ ]:
# --- 6. Adversarial validator epoch -------------------------------------------
import random, json, time
from ribosome.commit import CommitLedger, commitment_hash, join_candidates
from ribosome.data_targets import load_pool
from ribosome.generators import GAGenerator, StubGenerator
from ribosome.miner_logic import Synthetase
from ribosome.oracle import StubOracle
from ribosome.validator_logic import Chaperone

pool = load_pool()
ledger = CommitLedger(score_reveal_delay_b=4)
rng = random.Random(11)

def mk(hotkey, gen, strategy="honest"):
    return Synthetase(hotkey=hotkey, generator=gen, strategy=strategy,
                      k_candidates=4, seed=abs(hash(hotkey)) % 2**31)

miners = [mk(f"ga-{i}", GAGenerator()) for i in range(16)] \
       + [mk(f"stub-{i}", StubGenerator()) for i in range(8)] \
       + [mk(f"lazy-{i}", GAGenerator(), "lazy") for i in range(4)] \
       + [mk(f"leak-{i}", GAGenerator(), "leaker") for i in range(2)]

reveal_cache = {}
phases = {}
for epoch in range(4):
    if pool.should_rotate(epoch):
        pool.rotate(seed=epoch)
    t0 = time.perf_counter()
    for m in miners:
        tgt_id = m.act(epoch, pool, ledger)
    phases[f"epoch{epoch}-commit"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    for m in miners:
        m.reveal(epoch, ledger, epoch)
    phases[f"epoch{epoch}-reveal"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    ev = Chaperone(oracle=StubOracle()).evaluate_epoch(epoch, pool, ledger)
    phases[f"epoch{epoch}-evaluate"] = time.perf_counter() - t0

    if epoch == 3:
        groups = {}
        for h, e in ev.per_miner.items():
            g = h.split("-")[0]
            groups.setdefault(g, []).append(e)
        print(f"epoch {epoch} outcome summary:")
        for g, evs in sorted(groups.items()):
            acc = sum(e.accepted for e in evs)
            dup = sum(e.is_duplicate for e in evs)
            w = sum(ev.weights.get(e.hotkey, 0.0) for e in evs)
            print(f"  {g:6s} n={len(evs):2d} accepted={acc:2d} "
                  f"duplicate={dup:2d} weight={100*w:5.1f}%")
        # evidence assertions
        assert all(not e.accepted for e in groups["leak"]), "leaker accepted!"
        json.dump(
            {g: {"accepted": sum(e.accepted for e in evs),
                 "duplicates": sum(e.is_duplicate for e in evs),
                 "weight_share": sum(ev.weights.get(e.hotkey, 0.0) for e in evs)}
             for g, evs in groups.items()},
            open("validator_epoch_evidence.json", "w"), indent=2)
        print("saved validator_epoch_evidence.json")
print("phase timings:", {k: f"{v*1000:.0f}ms" for k, v in phases.items()})


In [ ]:
# --- 7. RhoFold+ gate (optional GPU 3D oracle) ---------------------------------
# If you attach the RhoFold+ repo + checkpoint as a Kaggle dataset, this cell
# wires the production RhoFoldOracle (PDB -> C3' contact map -> dot-bracket)
# and refolds one sequence on the GPU. Otherwise it prints a skip note and the
# notebook stays green with the 2D oracles.
import glob
from pathlib import Path

repo_c = [p for p in glob.glob("/kaggle/input/**/inference.py", recursive=True)]
ckpt_c = [p for p in glob.glob("/kaggle/input/**/*.pt", recursive=True)]
if repo_c and ckpt_c:
    from ribosome.oracle import RhoFoldOracle
    oracle = RhoFoldOracle(repo=str(Path(repo_c[0]).parent),
                           checkpoint=ckpt_c[0])
    seq = "GGGGAAACCCCAAAGGGGAAACCCCAAAGGGGAAACCCC"
    import time; t0 = time.perf_counter()
    db = oracle.fold(seq)
    print(f"RhoFold+ fold in {time.perf_counter()-t0:.1f}s -> {db[:40]}...")
else:
    print("RhoFold+ not attached - skipped (auto oracle uses ViennaRNA/Nussinov).")
    print("To enable: add a dataset with the RhoFold+ repo (inference.py) and")
    print("the RhoFold.pt checkpoint; this cell wires it automatically.")


## Take-aways

| item | result |
|---|---|
| EVALUATE budget | oracle throughput × 360 s covers hundreds of miners at K=4 |
| duplicate gate | exact 64-dim k-mer Jaccard on GPU, θ_dup = 0.85 catches ≥95% of 1–3-mutation clones, thousands of miners in ms |
| adversarial epoch | leakers 0%, clones zeroed, honest miners paid (evidence JSON saved) |
| RhoFold+ | auto-wired when the dataset is attached, else clean skip |

Next: `03_end_to_end_epoch_gpu.ipynb` — miner ⊕ validator in one 5-phase epoch.
